# Exp_2 — Encoder Fine-tuning (bge-small-en-v1.5)

Fine-tunes the retrieval encoder using **TripletLoss** on 622 hard-negative triplets from `consultas_centro_control.json`.
After fine-tuning, rebuilds the FAISS index and regenerates all PKLs (train/val/test) for `lora.ipynb`.

**Run on Colab T4 — ~30 min total.**

---
**Orden de pasos:**
1. Ajustar `DRIVE_PATH` en la celda de Config
2. Correr hasta la celda de Setup → **reiniciar el runtime**
3. Correr el resto del notebook de arriba a abajo

In [ ]:
# ── CELL 0: Config ──────────────────────────────────────────────────────────
DRIVE_PATH   = "/content/drive/MyDrive/MASTER/Tercer_semestre/NLP_2/Competencia"
ENCODER_BASE = "BAAI/bge-small-en-v1.5"
ENCODER_OUT  = "encoder_finetuned"   # directorio local en Colab
RETRIEVAL_K  = 8    # chunks para FAISS+BM25 candidate pool
PROMPT_K     = 3    # chunks que van al PKL (contexto para lora.ipynb)
RRF_K        = 60   # constante RRF estándar

In [ ]:
# ── CELL 1: Mount Drive + Install ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run([
    "pip", "install", "-q",
    "sentence-transformers>=3.0.0",
    "faiss-cpu>=1.7.4",
    "rank-bm25>=0.2.2",
    "tiktoken>=0.7.0",
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "datasets>=2.20.0",
    "scikit-learn>=1.3.0",
], check=True)

print("\n=== IMPORTANTE ===")
print("Reinicia el runtime ahora (Runtime → Restart runtime).")
print("Luego continúa desde la siguiente celda.")

In [ ]:
# ── CELL 2: Imports (correr DESPUÉS del restart) ─────────────────────────────
import os, json, re, shutil, unicodedata
import numpy as np
import pandas as pd
import torch
import faiss
from pathlib import Path
from rank_bm25 import BM25Okapi
from sklearn.model_selection import train_test_split
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import TripletLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── CELL 3: Copiar archivos desde Drive ──────────────────────────────────────
# Archivos a copiar desde Drive al directorio local de Colab
files_to_copy = [
    "retrieval_index.json",
    "Data/tools_definition.json",
    "Data/consultas_centro_control.json",
    "Data/train.csv",
    "Data/test.csv",
]

os.makedirs("Data", exist_ok=True)

for f in files_to_copy:
    src = os.path.join(DRIVE_PATH, f)
    dst = f
    if os.path.dirname(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f"  Copiado: {f}")
    else:
        print(f"  Ya existe: {f}")

# Knowledge base
kb_src = os.path.join(DRIVE_PATH, "Data/knowledge_base")
kb_dst = "Data/knowledge_base"
if not os.path.exists(kb_dst):
    shutil.copytree(kb_src, kb_dst)
    print("  Copiado: Data/knowledge_base/")
else:
    print("  Ya existe: Data/knowledge_base/")

print("\nArchivos locales listos.")

In [ ]:
# ── CELL 4: Cargar documentos KB (texto completo por doc_id) ─────────────────
docs = {}  # {doc_id: full_text}
kb_root = Path("Data/knowledge_base/knowledge_base")
for doc_path in sorted(kb_root.rglob("doc.md")):
    doc_id = doc_path.parent.name
    docs[doc_id] = doc_path.read_text(encoding="utf-8")

print(f"Documentos KB cargados: {len(docs)}")

def get_doc_text(doc_id: str) -> str:
    return docs.get(doc_id, "")

In [ ]:
# ── CELL 5: Cargar chunks + construir índice baseline ────────────────────────
with open("retrieval_index.json") as f:
    chunks = json.load(f)
print(f"Chunks cargados: {len(chunks)}")

# FAISS baseline (vectores originales de bge-small)
vecs_base = np.array([c["vector"] for c in chunks], dtype=np.float32)
DIM = vecs_base.shape[1]
index_base = faiss.IndexFlatIP(DIM)
index_base.add(vecs_base)
print(f"FAISS baseline: {index_base.ntotal} vectores, dim={DIM}")

# BM25 (basado en texto — el mismo para baseline y fine-tuned)
def _strip_accents(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

def _bm25_tok(text: str) -> list:
    return re.findall(r'\w+', _strip_accents(text).lower())

bm25_index = BM25Okapi([_bm25_tok(c["text"]) for c in chunks])
print("BM25 construido")

In [ ]:
# ── CELL 6: Funciones de retrieval y evaluación ──────────────────────────────
def encode_query(encoder, query: str) -> np.ndarray:
    """Encodifica query con el encoder dado (sentence-transformers)."""
    vec = encoder.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    return vec.astype(np.float32)

def retrieve_hybrid(query: str, faiss_idx, encoder, k: int = RETRIEVAL_K) -> list:
    n = len(chunks)
    k_cand = min(k * 4, n)

    # Dense FAISS
    q_vec = encode_query(encoder, query)
    _, dense_idxs = faiss_idx.search(q_vec, k_cand)
    dense_idxs = [i for i in dense_idxs[0] if i < n]

    # BM25
    bm25_scores = bm25_index.get_scores(_bm25_tok(query))
    bm25_idxs   = np.argsort(bm25_scores)[::-1][:k_cand].tolist()

    # Reciprocal Rank Fusion
    rrf = {}
    for rank, idx in enumerate(dense_idxs):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (RRF_K + rank + 1)
    for rank, idx in enumerate(bm25_idxs):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (RRF_K + rank + 1)

    top = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]
    return [chunks[i] for i in top]

def eval_pk(faiss_idx, encoder, consultas, ks=(1, 3, 5, 8)):
    """Compute P@k / R@k (1 doc relevante por query → idénticos)."""
    results = {}
    for k in ks:
        hits = 0
        for item in consultas:
            top = retrieve_hybrid(item["query"], faiss_idx, encoder, k=k)
            if item["doc_id"] in [c["doc_id"] for c in top]:
                hits += 1
        results[k] = hits / len(consultas)
    return results

print("Funciones definidas.")

In [ ]:
# ── CELL 7: Evaluar P@K baseline ────────────────────────────────────────────
with open("Data/consultas_centro_control.json") as f:
    consultas = json.load(f)
print(f"Consultas de evaluación: {len(consultas)}")

encoder_base_model = SentenceTransformer(ENCODER_BASE)
encoder_base_model.eval()

print("Evaluando baseline (bge-small-en-v1.5 sin fine-tuning)...")
baseline_metrics = eval_pk(index_base, encoder_base_model, consultas)

print("\nBaseline P@K (hybrid search):")
for k, v in baseline_metrics.items():
    print(f"  P@{k}: {v:.4f}")

In [ ]:
# ── CELL 8: Construir triplets (query, pos_doc, hard_neg_doc) ─────────────────
triplet_data = {"anchor": [], "positive": [], "negative": []}
skipped = 0

for item in consultas:
    if "hard_negative_doc_id" not in item:
        continue
    pos_text = get_doc_text(item["doc_id"])
    neg_text  = get_doc_text(item["hard_negative_doc_id"])
    if not pos_text or not neg_text:
        skipped += 1
        continue
    triplet_data["anchor"].append(item["query"])
    triplet_data["positive"].append(pos_text)
    triplet_data["negative"].append(neg_text)

train_dataset = Dataset.from_dict(triplet_data)
print(f"Triplets para entrenamiento: {len(train_dataset)} (saltados: {skipped})")
print(f"\nEjemplo:")
print(f"  Anchor:   {train_dataset[0]['anchor'][:100]}")
print(f"  Positive: {train_dataset[0]['positive'][:100]}")
print(f"  Negative: {train_dataset[0]['negative'][:100]}")

In [ ]:
# ── CELL 9: Fine-tuning del encoder con TripletLoss ──────────────────────────
# TripletLoss penaliza triplets donde dist(anchor,neg) < dist(anchor,pos) + margin
# Los hard negatives son documentos semánticamente cercanos pero incorrectos.
# El encoder aprende a separar el doc correcto del doc incorrecto para cada query.

model_ft = SentenceTransformer(ENCODER_BASE)

loss = TripletLoss(model=model_ft)

train_args = SentenceTransformerTrainingArguments(
    output_dir=ENCODER_OUT,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.10,
    fp16=True,
    evaluation_strategy="no",
    save_strategy="epoch",
    load_best_model_at_end=False,
    logging_steps=10,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model_ft,
    args=train_args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()

# Guardar modelo final
model_ft.save(ENCODER_OUT)
print(f"\nModelo guardado en: {ENCODER_OUT}/")

In [ ]:
# ── CELL 10: Reindexar con encoder fine-tuned + evaluar ─────────────────────
texts = [c["text"] for c in chunks]
print(f"Re-embebiendo {len(texts)} chunks con encoder fine-tuned...")

vecs_ft = model_ft.encode(
    texts,
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)

# Nuevo índice FAISS
index_ft = faiss.IndexFlatIP(vecs_ft.shape[1])
index_ft.add(vecs_ft)
print(f"FAISS fine-tuned: {index_ft.ntotal} vectores")

# Actualizar vectores en chunks (para guardar en retrieval_index.json)
for i, c in enumerate(chunks):
    c["vector"] = vecs_ft[i].tolist()

# Guardar archivos de índice actualizados
with open("retrieval_index.json", "w") as f:
    json.dump(chunks, f)
print("retrieval_index.json actualizado")

faiss.write_index(index_ft, "faiss.index")
print("faiss.index guardado")

# Evaluar fine-tuned
print("\nEvaluando encoder fine-tuned...")
ft_metrics = eval_pk(index_ft, model_ft, consultas)

print("\n=== Resultados: Baseline vs Fine-tuned ===")
print(f"{'k':>4} | {'Baseline':>10} | {'Fine-tuned':>10} | {'Delta':>8}")
print("-" * 42)
for k in [1, 3, 5, 8]:
    base = baseline_metrics[k]
    ft   = ft_metrics[k]
    delta_str = f"{ft - base:+.4f}"
    print(f"P@{k:2d} | {base:10.4f} | {ft:10.4f} | {delta_str:>8}")

In [ ]:
# ── CELL 11: Preprocesar train/val/test ──────────────────────────────────────
import unicodedata as ud

df_raw  = pd.read_csv("Data/train.csv")
df_test = pd.read_csv("Data/test.csv")
print(f"Raw: train={len(df_raw)}, test={len(df_test)}")

def normalize_query(q):
    return ud.normalize("NFC", str(q)).strip()

df_raw["query"] = df_raw["query"].apply(normalize_query)
df_raw = df_raw.drop_duplicates(subset=["query"])
print(f"Tras dedup: {len(df_raw)}")

with open("Data/tools_definition.json") as f:
    tools_def_raw = json.load(f)
TOOLS_DEF   = {t["name"]: t for t in tools_def_raw["tools"]}
VALID_TOOLS = set(TOOLS_DEF.keys())

def tool_name(tc):
    return str(tc).split("(")[0].strip()

df_raw = df_raw[df_raw["tool_call"].apply(tool_name).isin(VALID_TOOLS)]
print(f"Tras validación: {len(df_raw)}")

df_train, df_val = train_test_split(
    df_raw, test_size=0.10, random_state=42,
    stratify=df_raw["tool_call"].apply(tool_name)
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test["query"] = df_test["query"].apply(normalize_query)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

In [ ]:
# ── CELL 12: Regenerar PKLs con hybrid search (encoder fine-tuned) ───────────
from tqdm.auto import tqdm

def retrieve_for_pkl(query: str) -> list:
    return retrieve_hybrid(query, index_ft, model_ft, k=PROMPT_K)

for split_name, df_split in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"Procesando {split_name} ({len(df_split)} queries)...")
    df_split = df_split.copy()
    df_split["context_chunks"] = [
        retrieve_for_pkl(q)
        for q in tqdm(df_split["query"].tolist(), desc=split_name)
    ]
    pkl_name = f"{split_name}_processed.pkl"
    df_split.to_pickle(pkl_name)
    print(f"  Guardado: {pkl_name} ({len(df_split)} filas)")

    # Actualizar variable
    if split_name == "train":
        df_train = df_split
    elif split_name == "val":
        df_val = df_split
    else:
        df_test = df_split

print("\nPKLs generados correctamente.")

In [ ]:
# ── CELL 13: Guardar todo en Drive ───────────────────────────────────────────
files_to_save = [
    "retrieval_index.json",
    "faiss.index",
    "train_processed.pkl",
    "val_processed.pkl",
    "test_processed.pkl",
]

for f in files_to_save:
    dst = os.path.join(DRIVE_PATH, f)
    shutil.copy(f, dst)
    size_mb = os.path.getsize(f) / 1e6
    print(f"  Guardado: {f} ({size_mb:.1f} MB) → Drive")

# Guardar encoder fine-tuned
encoder_dst = os.path.join(DRIVE_PATH, ENCODER_OUT)
if os.path.exists(encoder_dst):
    shutil.rmtree(encoder_dst)
shutil.copytree(ENCODER_OUT, encoder_dst)
print(f"  Guardado: {ENCODER_OUT}/ → {encoder_dst}/")

print("\n✓ Todo guardado en Drive.")
print("\nSiguiente paso: abrir lora.ipynb en Colab con los nuevos PKLs.")
print("Los PKLs generados usan el encoder fine-tuned → retrieval mejorado.")